## API DEVELOPMENT
### PROJECT TITLE
### Build Your First API (Sales Data API)

#### GOAL

👉 Create an API that:
- Serves data from database
- Returns results in JSON
- Can be used by frontend / apps

### WHAT IS AN API?

👉 API = bridge between systems

**Example:**

App → asks data
API → gives response

Like:
```
Client → API → Database → API → Client
```
### WHAT IS JSON?

👉 JSON = data format used in APIs

**Example:**
```
{
  "name": "Ali",
  "marks": 80
}
```

### TECHNOLOGY
- Python
- Flask
- SQLite
- Pandas

### STEP 0 — INSTALL FLASK

In [1]:
pip install flask

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\DELL XPS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Create Project Folder

Create a new folder anywhere:
```
SalesAPI
```
Open this folder in VS Code / Notepad.

### Step 1 — Basic Flask App
**File:** app.py
```
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to Sales API"

if __name__ == "__main__":
    app.run(debug=True)
```
#### Explanation
- Flask() → Creates the app.
- @app.route("/") → Defines a URL endpoint (/).
- app.run() → Starts server at http://127.0.0.1:5000/.

👉 Run:
```
python app.py
```
Open browser → http://127.0.0.1:5000/

### Step 2 — Connect Database
Add to app.py
```
import sqlite3

def get_db_connection():
    conn = sqlite3.connect("sales.db")
    conn.row_factory = sqlite3.Row
    return conn
```
#### Why?
- Connects to sales.db.
- row_factory → Converts rows into dictionary format (so they can be JSON).


### Step 3 — Create API Endpoint (GET Data)
Add to app.py
```
from flask import jsonify

@app.route("/sales", methods=["GET"])
def get_sales():
    conn = get_db_connection()
    
    query = """
    SELECT customers.name, products.product_name, orders.quantity,
    (products.price * orders.quantity) AS total
    FROM orders
    JOIN customers ON orders.customer_id = customers.id
    JOIN products ON orders.product_id = products.id
    """
    
    data = conn.execute(query).fetchall()
    conn.close()
    
    sales_list = [dict(row) for row in data]
    
    return jsonify(sales_list)
```
#### Explanation
- /sales → API endpoint.
- GET → Fetches data.
- jsonify() → Converts Python list → JSON.

👉 Open: http://127.0.0.1:5000/sales

Output Example:
```
[
  {
    "name": "Ali",
    "product_name": "Laptop",
    "quantity": 1,
    "total": 100000
  }
]
```

### Step 4 — Add Filter API
Add to app.py
```
@app.route("/sales/<city>", methods=["GET"])
def get_sales_by_city(city):
    conn = get_db_connection()
    
    query = """
    SELECT customers.name, products.product_name, orders.quantity,
    (products.price * orders.quantity) AS total
    FROM orders
    JOIN customers ON orders.customer_id = customers.id
    JOIN products ON orders.product_id = products.id
    WHERE customers.city = ?
    """
    
    data = conn.execute(query, (city,)).fetchall()
    conn.close()
    
    return jsonify([dict(row) for row in data])
```
#### Explanation
- Dynamic route /sales/<city>.
- Example: http://127.0.0.1:5000/sales/Lahore
- Filters data by city.

Output Example:
```
[
  {
    "name": "Ali",
    "product_name": "Laptop",
    "quantity": 1,
    "total": 100000
  }
]
```

### Step 5 — Add New Data (POST API)
Add to app.py
```
from flask import request

@app.route("/add_order", methods=["POST"])
def add_order():
    data = request.json
    
    customer_id = data["customer_id"]
    product_id = data["product_id"]
    quantity = data["quantity"]
    
    conn = get_db_connection()
    
    conn.execute(
        "INSERT INTO orders (customer_id, product_id, quantity) VALUES (?, ?, ?)",
        (customer_id, product_id, quantity)
    )
    
    conn.commit()
    conn.close()
    
    return {"message": "Order added successfully"}
```
#### Explanation
- /add_order → API endpoint.
- POST → Adds new data.
- request.json → Reads JSON input from client.
- INSERT INTO → Saves new order in database.

👉 Example POST request:
```
{
  "customer_id": 1,
  "product_id": 2,
  "quantity": 3
}
```
Response:
```
{"message": "Order added successfully"}
```

http://127.0.0.1:5000/add_order

### Summary of Files & Steps
# Summary of Files & Steps

| Step | File      | What's Added              | Purpose                      |
|------|-----------|---------------------------|------------------------------|
| 0    | Terminal  | pip install flask         | Install Flask                |
| 1    | app.py    | Basic Flask app           | Start server                 |
| 2    | app.py    | DB connection function    | Connect SQLite               |
| 3    | app.py    | /sales endpoint           | Fetch sales data             |
| 4    | app.py    | /sales/ endpoint          | Filter by city               |
| 5    | app.py    | /add_order endpoint       | Insert new order             |